# Loading data

In [14]:
import pickle
import numpy as np
import pandas as pd
import os

In [31]:
df_data = pd.read_parquet('c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/Datasets/DiaTrend/Glucose_measurements_v3_DiaTrend.parquet')

# Filter valid rows (15min is not NaT)
df_valid_rows = df_data[df_data['15min'].notna()]

# Group by 'Patient_ID' and convert 'Measurement' to numpy array
patient_dict = {
    pid: group['Measurement'].to_numpy()  # Convert to int32
    for pid, group in df_valid_rows.groupby('Patient_ID')
}
print(f"Number of patients with valid data: {len(patient_dict)}")

Number of patients with valid data: 54


In [35]:
df_data[df_data['Patient_ID'] == '2']

,Patient_ID,Measurement_date,Measurement_time,Measurement,15min,5min
652364,2,2015-12-04,0 days 06:41:23,95.0,NaT,2015-12-04 06:40:00
652365,2,2015-12-04,0 days 06:46:22,107.0,2015-12-04 06:45:00,2015-12-04 06:45:00
652366,2,2015-12-04,0 days 06:51:22,110.0,NaT,2015-12-04 06:50:00
652367,2,2015-12-04,0 days 06:56:27,115.0,NaT,2015-12-04 06:55:00
652368,2,NaT,NaT,NaN,2015-12-04 07:00:00,2015-12-04 07:00:00
...,...,...,...,...,...,...
1255404,2,2021-06-06,0 days 23:54:06,401.0,NaT,NaT
1255405,2,2021-06-06,0 days 23:54:06,401.0,NaT,2021-06-06 23:55:00
1255406,2,2021-06-06,0 days 23:58:21,357.0,NaT,NaT
1255407,2,2021-06-06,0 days 23:59:06,387.0,NaT,NaT


In [6]:


# Define the file path (Replace with your filepath)
file_path_train = "C:/Users/Ciro/Mi Unidad (PDI)/_UGR_PDI_2024-05-30/RELIEF-T1D/Python_projects/HeterogeneityInfluence/Dataset/Preprocessed data/measurements_by_patient_train.pkl"
file_path_test = "C:/Users/Ciro/Mi Unidad (PDI)/_UGR_PDI_2024-05-30/RELIEF-T1D/Python_projects/HeterogeneityInfluence/Dataset/Preprocessed data/measurements_by_patient_test.pkl"
file_path_val = "C:/Users/Ciro/Mi Unidad (PDI)/_UGR_PDI_2024-05-30/RELIEF-T1D/Python_projects/HeterogeneityInfluence/Dataset/Preprocessed data/measurements_by_patient_val.pkl"

# Open files
with open(file_path_train, 'rb') as file:
    train_dict = pickle.load(file)

with open(file_path_test, 'rb') as file:
    test_dict = pickle.load(file)

with open(file_path_val, 'rb') as file:
    val_dict = pickle.load(file)
    
# data_dict = {}
# data_dict.update(train_dict)
# data_dict.update(val_dict)
# data_dict.update(test_dict)
# len(data_dict)

691

In [33]:
# Windows creation parameters
history_length = 8  # 120 min
horizons = [2, 4, 6, 8, 12, 16]  # 2 for 30 min, 4 for 60 min, etc.

path = 'c:/Users/Ciro/C/_UGR_PDI/RELIEF-T1D/windows/no_split_2025_DiaTrend/'
if not os.path.exists(path):
    # Si no existe, crearlo
    os.makedirs(path)


# Auxiliary function
def get_windows_one_step_walk_forward(bgl_measurement_dict, 
                                      history_length, horizon, 
                                      deduplicate_intra_patient=True) -> pd.DataFrame:
    """
      :param bgl_measurement_dict: Dictionary with patient_id as key and numpy array of temporal series of BGL measurements
      :param history_length: Number of samples used to predict (8 for 120 minutes)
      :param horizon: Prediction horizon in number of windows (2 for 30 minutes, 4 for 60 minutes, and so on)
      :return: Get windows without missing values from one step sliding windows
    """
    df_all_patient_set = pd.DataFrame()

    for patient_id, patient_series in bgl_measurement_dict.items():
        # print(f"Clave: {patient_id} — Valor: {patient_series}")

        # Creating the one step sliding window
        x = np.lib.stride_tricks.sliding_window_view(patient_series[:-horizon], history_length)
        y = np.lib.stride_tricks.sliding_window_view(patient_series[history_length:], horizon)

        # Removing rows with missing values (NaN)
        nan_rows_x = np.isnan(x).any(axis=1)  # In input values
        nan_rows_y = np.isnan(y).any(axis=1)  # In output values
        x = x[~(nan_rows_x | nan_rows_y)]  # Remove rows with NaN in either x or y from x
        y = y[~(nan_rows_x | nan_rows_y)]  # Remove rows with NaN in either x or y from y

        # Create DataFrame for the current patient
        df_x = pd.DataFrame(x)  # Input windows
        df_y = pd.DataFrame(y[:, -1])  # Output windows (last column of y)
        df_set = pd.concat([df_x, df_y], axis=1)  # Combine inputs and outputs
        df_set = df_set.astype(int)  # Convert to integer type

        # Renaming columns and adding patient_id.
        # Input: x0 a x7, Output: y
        new_cols = [f"x{i}" for i in range(8)] + ["y"]
        df_set.columns = new_cols
        df_set['patient_id'] = patient_id
        
        # Concat the current patient's DataFrame to the main DataFrame
        df_all_patient_set = pd.concat([df_all_patient_set, df_set], ignore_index=True)
    
    if deduplicate_intra_patient:
        # Drop duplicated instances in a patient. All columns are considered for duplication, therefore, if a patient has the same input and output values, it will be considered a duplicate, but if there are two equal instances but from different patients, they will not be considered duplicates. 
        df_all_patient_set = df_all_patient_set.drop_duplicates()
        df_all_patient_set.reset_index(drop=True, inplace=True)
    
    return df_all_patient_set


for current_horizon in horizons:
    print(f"Processing horizon: {current_horizon}")
    df_windows = get_windows_one_step_walk_forward(bgl_measurement_dict=patient_dict, 
                                                   history_length=history_length, 
                                                   horizon=current_horizon, 
                                                   deduplicate_intra_patient=True)
    
        # NOTE: Replace with your filepath
    df_windows.to_parquet(f'{path}/train_windows_horizon_{current_horizon}.parquet')
    print(f"Saved windows for horizon {current_horizon} to parquet files.")
    print("--------------------------------------------------")
    print()


Processing horizon: 2
Saved windows for horizon 2 to parquet files.
--------------------------------------------------

Processing horizon: 4
Saved windows for horizon 4 to parquet files.
--------------------------------------------------

Processing horizon: 6
Saved windows for horizon 6 to parquet files.
--------------------------------------------------

Processing horizon: 8
Saved windows for horizon 8 to parquet files.
--------------------------------------------------

Processing horizon: 12
Saved windows for horizon 12 to parquet files.
--------------------------------------------------

Processing horizon: 16
Saved windows for horizon 16 to parquet files.
--------------------------------------------------
